In [ ]:
!pip install transformers
!pip install qwen-vl-utils[decord]
!pip install accelerate


In [ ]:
from google.colab import files

uploaded = files.upload()  # Choose your local video file (5–10 sec)
video_path = list(uploaded.keys())[0]
print("Uploaded video:", video_path)


Saving v28.mov to v28 (1).mov
Uploaded video: v28 (1).mov


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

# Load model (uses GPU automatically)
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct", dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

# Prepare message for video explanation
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path,  # local video file
                "fps": 1.0,           # sample 1 frame per second
                "max_pixels": 256*32*32,
            },
            {"type": "text", "text": "Explain what is happening in this video clip."}
        ],
    }
]

# Process input
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
images, videos, video_kwargs = process_vision_info(messages, image_patch_size=16, return_video_kwargs=True, return_video_metadata=True)

if videos is not None:
    videos, video_metadatas = zip(*videos)
    videos, video_metadatas = list(videos), list(video_metadatas)
else:
    video_metadatas = None

inputs = processor(
    text=text,
    images=images,
    videos=videos,
    video_metadata=video_metadatas,
    return_tensors="pt",
    do_resize=False,
    **video_kwargs
)
inputs = inputs.to(model.device)

# Generate output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print("\n=== Video Explanation ===\n")
print(output_text)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]


=== Video Explanation ===

["A white car is driving through an intersection when it suddenly collides with a dark green SUV. The impact causes the white car to spin out of control and skid across the road, leaving a trail of debris. The SUV, which was turning left, appears to have been the cause of the collision. The scene is set on a wet road, likely after a rain, with traffic lights and houses visible in the background. The accident occurs near a traffic signal, and the white car ends up in a damaged state, with its rear end visibly crushed. The dark green SUV continues forward, seemingly unharmed, while the white car's"]


In [1]:
import os
from openai import OpenAI

client = OpenAI(
    # API keys for the Singapore and China (Beijing) regions are different. Get an API key: https://www.alibabacloud.com/help/en/model-studio/get-an-api-key
    # If the environment variable is not configured, replace the following line with your Model Studio API key: api_key="sk-xxx",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # The following is the base_url for the Singapore region. If you use a model in the China (Beijing) region, replace the base_url with: https://dashscope.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3-vl-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241115/cqqkru/1.mp4"
                    }
                },
                {
                    "type": "text",
                    "text": "What is the content of this video?"
                }
            ]
        }
    ]
)

print(completion.choices[0].message.content)

PermissionDeniedError: Error code: 403 - {'error': {'code': 'AccessDenied.Unpurchased', 'param': None, 'message': 'Access to model denied. Please make sure you are eligible for using the model.', 'type': 'AccessDenied.Unpurchased'}, 'id': 'chatcmpl-8e40367a-7bdc-9f83-b71d-74c077762fef', 'request_id': '8e40367a-7bdc-9f83-b71d-74c077762fef'}